In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Parameters ----------------
start = 1e-1        
stop = 1e3         
λ_ovr_g = np.logspace(np.log10(start), np.log10(stop), num=40, endpoint=True)
λ_ovr_m = np.array([0.1, 1.0, 10.0, 100.0, 1000.0])

N = 13
N_θ = 10001

# ---------------- Storage ----------------
expval_Pop_0 = np.zeros((len(λ_ovr_m), len(λ_ovr_g)))
expval_n_sqr_0 = np.zeros((len(λ_ovr_m), len(λ_ovr_g)))

# ---------------- Loop ----------------
for i, λ_m in enumerate(λ_ovr_m):
    for j, λ_g in enumerate(λ_ovr_g):

        print("Computing for λ/m =", λ_m, ", λ/g =", λ_g)
        (
            E_ν_vector, 
            exp_plusiθ_op_matrix, 
            exp_minusiθ_op_matrix, 
            nθ_op_matrix, 
            nθ_sqr_op_matrix
        ) = single_link_data(λ_m, λ_g, N, N_θ)
        """
        H_mathieu = H_0 + H_additional

        H_0 = Σ_l H_l, with H_l = Σ_l [ 2/(λ/m) + 1/(λ/g) ] n_l^2 - cos(θ_l), l={12,24,13,34} 
        have analytical solution obdatied in single_link_data(), so in Mathieu basis are 
        diagonal with same eigenvalues E_ν_vector for each link.

        H_additional = 2/(λ/m) * (n12 * n13 + n24 * n34 - n12 * n24 - n13 * n34 )
        """
        H_0_l = sp.diags(E_ν_vector, offsets = 0, format='csr')
        id_N = sp.eye(N, format='csr')
        H_0 = (
              kron_n(H_0_l, id_N, id_N, id_N) 
            + kron_n(id_N, H_0_l, id_N, id_N) 
            + kron_n(id_N, id_N, H_0_l, id_N) 
            + kron_n(id_N, id_N, id_N, H_0_l)
        )

        Udiff = exp_plusiθ_op_matrix.conj().T - exp_minusiθ_op_matrix
        print("exponentials are nice? - Error = ", np.max(np.abs(Udiff)))
        D = nθ_op_matrix - nθ_op_matrix.conj().T
        print("single-link max asymmetry:", np.max(np.abs(D)))
        D2 = nθ_sqr_op_matrix - nθ_sqr_op_matrix.conj().T
        print("single-link max asymmetry:", np.max(np.abs(D2)))

        # Choice of order is important for Kronecker products. Here we take [n12, n24, n13, n34]
        H_additional = (2.0/λ_m) * (
            kron_n(nθ_op_matrix, id_N, nθ_op_matrix, id_N) # n12 * n13
          + kron_n(id_N, nθ_op_matrix, id_N, nθ_op_matrix) # n24 * n34
          - kron_n(nθ_op_matrix, nθ_op_matrix, id_N, id_N) # n12 * n24
          - kron_n(id_N, id_N, nθ_op_matrix, nθ_op_matrix) # n13 * n34
        )
        H_mathieu = H_0 + H_additional

        diff = H_mathieu - H_mathieu.getH()
        print("nnz:", diff.nnz)
        print("data size:", diff.data.size)
        if diff.data.size > 0:
            print("max |diff|:", np.max(np.abs(diff.data)))
        else:
            print("diff is exactly zero (sparse)")

        is_hermitian = (H_mathieu - H_mathieu.getH()).nnz == 0
        print("Hermitian?", is_hermitian)
        assert sp.isspmatrix(H_mathieu)
        off_diag_nnz_mathieu = (H_mathieu - sp.diags(H_mathieu.diagonal(), format="csr")).nnz
        print("Off-diagonal elements:", off_diag_nnz_mathieu)
        print("Shape of the Hamiltonian H_mathieu -> ", H_mathieu.shape)
        
        E, ψ = spla.eigsh(H_mathieu, k=1, which="SA", tol=1e-8, maxiter=100000)
        ψ_chosen = ψ[:, 0]

        P = 0.5 * (
            kron_n(
                exp_plusiθ_op_matrix, 
                exp_plusiθ_op_matrix, 
                exp_minusiθ_op_matrix, 
                exp_minusiθ_op_matrix
            ) + kron_n(
                exp_minusiθ_op_matrix, 
                exp_minusiθ_op_matrix, 
                exp_plusiθ_op_matrix, 
                exp_plusiθ_op_matrix
            )
        )
        n_sqr = kron_n(nθ_sqr_op_matrix, id_N, id_N, id_N)

        expval_Pop_0[i, j] =  np.vdot(ψ_chosen, P @ ψ_chosen).real
        expval_n_sqr_0[i, j] =  np.vdot(ψ_chosen, n_sqr @ ψ_chosen).real

In [ ]:
filename = (
    f"plaquette_reduced4vars_alphas0_mathieubasis_"
    f"expvals_Pop_nsqr_data_several_λ_ovr_m_λ_ovr_g_0_1_to_1000_N_{N}_Ntheta_{N_θ}_mytake.npz"
)
np.savez_compressed(
    filename,
    lambda_over_m = λ_ovr_m, 
    lambda_over_g = λ_ovr_g,
    expvals_plaquette_operator=expval_Pop_0,
    expvals_n12_sqr=expval_n_sqr_0,
    N=N,
    N_θ=N_θ
)
print(f"\nResults saved to {filename}")